In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/q20_rewrite/checkpoints/post_cell_19_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 20 ###

### Updated Code ###

def add_gap_rows_2(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop=True)

    # Create gaps DataFrame
    gaps = (
        df_essay[df_essay['gap_length'] > 0]
        .assign(
            start=lambda x: x['discourse_end'].shift(1).fillna(0) + 1,
            end=lambda x: x['discourse_start'] - 1,
            discourse_type='Nothing',
            gap=np.nan,
            gap_end=np.nan
        )[['start', 'end', 'discourse_type', 'gap', 'gap_end']]
    )

    # Add gap at end if required
    last_row = df_essay.iloc[-1]
    if last_row['gap_end_length'] > 0:
        end_gap_data = {
            'start': last_row['discourse_end'] + 1,
            'end': last_row['discourse_end'] + last_row['gap_end_length'],
            'discourse_type': 'Nothing',
            'gap': np.nan,
            'gap_end': np.nan
        }
        gaps = pd.concat([gaps, pd.DataFrame([end_gap_data])], ignore_index=True)

    df_essay = pd.concat([df_essay, gaps]).sort_values(by='discourse_start').reset_index(drop=True)
    return df_essay


def print_colored_essay(essay):
    df_essay = add_gap_rows_2(essay)
    essay_file = f"/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"

    ents = df_essay.apply(
        lambda row: {'start': int(row['discourse_start']), 'end': int(row['discourse_end']), 'label': row['discourse_type']},
        axis=1
    ).tolist()

    with open(essay_file, 'r') as file:
        data = file.read()

    doc2 = {
        "text": data,
        "ents": ents,
    }

    colors = {
        'Lead': '#EE11D0',
        'Position': '#AB4DE1',
        'Claim': '#1EDE71',
        'Evidence': '#33FAFA',
        'Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow',
        'Rebuttal': 'red'
    }
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}

print_colored_essay(IREWR_plug_2)

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/q20_rewrite/checkpoints/post_cell_20_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_20_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[20], f)


In [ ]:
opt_output = Out.get(4)